In [ ]:
# Notebooks live in Notebooks/, but every data path is relative to the repo root.
import os, sys
while not os.path.isdir("data") and os.getcwd() != "/":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

# GRUM: one model across contracts, instead of one Bradley-Terry fit per contract

Formalization: [docs/grum_formalization.md](../docs/grum_formalization.md). Read it first —
it defines the mapping (a **contract** plays the role of a GRUM **agent**) and says why the
MC-EM machinery in `grum4llm/` is not needed here.

The model:

    U_ij = delta_j + x_i^T B z_j + eps_ij

with `i` = experiment condition (the contract in the prompt), `j` = laptop.
Because we observe a **continuous margin per pair** and not just a ranking, this collapses to

    y = gamma_i + (a + B^T x_i)^T dz + noise,     dz = z_j - z_j'

i.e. **one OLS** with `dz (x) x` interaction columns. The per-condition BT weight vector is
`w_i = a + B^T x_i`.

What the parameter blocks mean:

| block | meaning | existing finding it maps to |
|---|---|---|
| `delta` (= `a`) | preference with **no** contract (`x = 0`) | Apple is top brand everywhere |
| **diagonal blocks** of `B` | **compliance** — contract on a feature moves that feature | +13.78 on `14-inch` when asked |
| **off-diagonal blocks** of `B` | **leakage** — contract on ram moves *brand* | the Apple collapse (C6) |
| `x`-conjunction | two constraints are worth more than the sum | R² jump under the double constraint |
| `gamma_i` | positional bias — elicitation artifact, not utility | -5 to -11 |

The whole thesis conjecture reduces to: **are the off-diagonal blocks of `B` non-zero?**

In [ ]:
import json, glob
import numpy as np
import pandas as pd
import statsmodels.api as sm

BRAND  = ["ASUS", "Apple", "Dell", "HP", "Lenovo"]
SCREEN = ["13-inch", "14-inch", "16-inch"]
RAM    = ["4GB", "8GB", "16GB"]
LEVELS = {"brand": BRAND, "screen": SCREEN, "ram": RAM}

# The (feature, level) pairs a contract can name. Collected so far: only these two.
# "no constraint on this feature" = 0; "no contract at all" = the all-zero vector,
# which is what makes delta mean "preference with no contract".
X_COLS = [("screen", "14-inch"), ("ram", "8GB")]

MODELS = [("qwen", "0.5"), ("qwen", "7"), ("qwen", "32"), ("qwen", "72"),
          ("gemma", "1"),  ("gemma", "4"), ("gemma", "12"), ("gemma", "27")]
# qwen-0.5B has no measurable preference on anything; kept as a noise floor, read with care.
REAL = [m for m in MODELS if m != ("qwen", "0.5")]


def effects_code(series, levels):
    """Deviation (sum-to-zero) coding: p levels -> p-1 columns, last level = -1.
    Every level ends up with a coefficient, and the gauge is explicit instead of
    hidden inside statsmodels' pinv fallback."""
    out = np.zeros((len(series), len(levels) - 1))
    idx = pd.Series(series).map({lv: k for k, lv in enumerate(levels)}).values
    for k in range(len(levels) - 1):
        out[:, k] = (idx == k).astype(float)
    out[idx == len(levels) - 1, :] = -1.0
    return out


def z_names(features):
    return [f"{f}_{lv}" for f in features for lv in LEVELS[f][:-1]]


def restore_dropped(coef, features):
    """Recover the last level of each block from the sum-to-zero constraint."""
    w = dict(zip(z_names(features), coef))
    for f in features:
        w[f"{f}_{LEVELS[f][-1]}"] = -sum(w[f"{f}_{lv}"] for lv in LEVELS[f][:-1])
    return w


def pair_id(df):
    """Unordered item pair - the clustering unit. Each appears in 10 rows
    (2 orders x 5 templates), so treating rows as independent overstates
    confidence ~3x (progress log, 2026-08-09)."""
    a = df.a_brand + "|" + df.a_screen + "|" + df.a_ram
    b = df.b_brand + "|" + df.b_screen + "|" + df.b_ram
    return np.where(a < b, a + "//" + b, b + "//" + a)


def load_runs(family, size):
    """All conditions for one model. config.json is the only source of truth."""
    runs = {}
    for cfg_path in glob.glob("data/laptops_robustness*/*/config.json"):
        c = json.load(open(cfg_path))
        if (c.get("model_family") != family or str(c.get("model_size")) != str(size)
                or c.get("alternatives") != "laptops_robustness"):
            continue
        cid = c.get("constraints_id")
        if cid in runs:            # gemma sets are duplicated across two folders
            continue
        df = pd.read_csv(os.path.join(os.path.dirname(cfg_path), "scores.csv"))
        runs[cid] = (c.get("constraints", {}), df)
    return runs


def contract_vector(constraints, conjunction=True):
    x = np.array([1.0 if constraints.get(f) == lv else 0.0 for (f, lv) in X_COLS])
    return np.append(x, np.prod(x)) if conjunction else x

## 1. The pooled fit

One OLS per model, stacking every condition. Columns are
`[condition dummies | dz | dz*x for each contract component]`.

In [ ]:
def fit_grum(runs, features=("brand", "screen", "ram"), spec_tied=False,
             conjunction=True, cluster=True):
    """Pooled GRUM. Returns (results, conditions, feature-level names).

    spec_tied: keep only pairs where screen and ram are both tied, so brand is the
        only difference. Under a lexicographic rule that is the only place the brand
        utility is identified at all (see docs/grum_formalization.md section 5).
    """
    features, conds = list(features), list(runs)
    zn = z_names(features)
    Y, parts, groups = [], [], []
    for cid in conds:
        constraints, df = runs[cid]
        if spec_tied:
            df = df[(df.a_screen == df.b_screen) & (df.a_ram == df.b_ram)]
        dz = np.hstack([effects_code(df[f"a_{f}"], LEVELS[f])
                        - effects_code(df[f"b_{f}"], LEVELS[f]) for f in features])
        x = contract_vector(constraints, conjunction)
        g = np.zeros((len(df), len(conds)))
        g[:, conds.index(cid)] = 1.0                       # gamma_i, positional bias
        parts.append(np.hstack([g, dz] + [dz * xk for xk in x]))
        Y.append((df.score_a - df.score_b).values)
        groups.append(pair_id(df))
    X, Y = np.vstack(parts), np.concatenate(Y)
    model = sm.OLS(Y, X)
    res = (model.fit(cov_type="cluster", cov_kwds={"groups": np.concatenate(groups)})
           if cluster else model.fit())
    return res, conds, zn


def blocks(res, conds, zn, features=("brand", "screen", "ram")):
    """Split the coefficient vector into delta and the rows of B."""
    ng, nz = len(conds), len(zn)
    out = {"delta": restore_dropped(res.params[ng:ng + nz], features)}
    labels = [f"{f}={lv}" for (f, lv) in X_COLS] + ["conjunction"]
    for r, lab in enumerate(labels):
        seg = res.params[ng + nz * (1 + r): ng + nz * (2 + r)]
        if len(seg) == nz:
            out[f"B[{lab}]"] = restore_dropped(seg, features)
    return out

### Check: the pooled GRUM *is* the per-condition fits, reparameterized

If `w_i = a + B^T x_i` is right, then a saturated `x` must reproduce the separate
per-condition Bradley-Terry weights exactly. Dropping the conjunction term imposes
`w_{14+8} = w_{14} + w_{8} - w_{none}`, which should visibly fail.

In [ ]:
rows = []
for family, size in MODELS:
    runs = load_runs(family, size)
    if len(runs) < 4:
        continue
    # the current methodology: one fit per condition
    per_cond = {}
    for cid, (_, df) in runs.items():
        dz = np.hstack([effects_code(df[f"a_{f}"], LEVELS[f])
                        - effects_code(df[f"b_{f}"], LEVELS[f]) for f in LEVELS])
        r = sm.OLS((df.score_a - df.score_b).values, sm.add_constant(dz)).fit()
        per_cond[cid] = restore_dropped(r.params[1:], list(LEVELS))

    row = {"model": f"{family}-{size}B"}
    for conj in (False, True):
        res, conds, zn = fit_grum(runs, conjunction=conj, cluster=False)
        ng, nz = len(conds), len(zn)
        a, B = res.params[ng:ng + nz], res.params[ng + nz:].reshape(-1, nz)
        worst = max(
            max(abs(v - per_cond[cid][k]) for k, v in
                restore_dropped(a + contract_vector(c, conj) @ B, list(LEVELS)).items())
            for cid, (c, _) in runs.items())
        row["R2 additive-in-x" if not conj else "R2 saturated"] = round(res.rsquared, 4)
        row["max err additive" if not conj else "max err saturated"] = worst
    rows.append(row)

chk = pd.DataFrame(rows).set_index("model")
chk["max err saturated"] = chk["max err saturated"].map(lambda v: f"{v:.1e}")
chk["max err additive"] = chk["max err additive"].round(2)
print("max |pooled-implied weight - per-condition weight|, in log-odds\n")
print(chk.to_string())

Saturated: agreement to ~1e-14, i.e. machine precision — **the GRUM is a reparameterization of
what we already do, not a different estimator.** So nothing already established is put at risk
by switching to it.

Additive-in-`x`: off by up to ~2.8 log-odds, all of it at the double constraint. The
**conjunction term is real** and has to stay — it is the same effect as the R² jump under
`14in+8GB` noted on 2026-08-09.

## 2. The interaction matrix `B`

Rows are contract components, columns are the item-feature weights they shift.
`B` is not square (11 x 8), so it has no diagonal entries - it has **blocks**, grouped by
feature. **Diagonal blocks** (contract on a feature moves that same feature) are compliance;
**off-diagonal blocks** (contract on ram moves brand) are leakage.

In [ ]:
ORDER = ([f"brand_{b}" for b in BRAND] + [f"screen_{s}" for s in SCREEN]
         + [f"ram_{r}" for r in RAM])

for family, size in [("qwen", "32"), ("gemma", "27")]:
    runs = load_runs(family, size)
    res, conds, zn = fit_grum(runs)
    tab = pd.DataFrame(blocks(res, conds, zn)).T[ORDER]
    print(f"\n=== {family}-{size}B " + "=" * 60)
    print(tab.round(2).to_string())

Read `B[ram=8GB]` across the row: the contract puts **+9.5** on `ram_8GB` (compliance, the
its own diagonal block) — and at the same time **-2.95 on `brand_Apple`** while pushing every other brand up.
It was never asked about brand.

That `ram -> brand` off-diagonal block is the Apple collapse.

## 3. Is the leakage real? — and measured where brand is actually operative

Two versions. On **all pairs**, the brand block is diluted by the 6750 pairs that RAM decides
outright (the lexicographic finding of 2026-08-10). On **spec-tied** pairs — same screen, same
ram, so brand is the only difference — brand is the only thing that can decide, which under a
priority rule is the only place it is identified.

SEs clustered by unordered item pair throughout.

In [ ]:
def leakage_table(spec_tied):
    feats = ["brand"] if spec_tied else ["brand", "screen", "ram"]
    out = []
    for family, size in MODELS:
        runs = load_runs(family, size)
        if len(runs) < 4:
            continue
        res, conds, zn = fit_grum(runs, features=feats, spec_tied=spec_tied)
        ng, nz = len(conds), len(zn)
        bl = blocks(res, conds, zn, feats)

        # Wald test: the whole brand block of one B row is zero
        bcols = [k for k, nm in enumerate(zn) if nm.startswith("brand_")]
        pv = []
        for r in (0, 1):
            R = np.zeros((len(bcols), len(res.params)))
            for i, c in enumerate(bcols):
                R[i, ng + nz * (1 + r) + c] = 1.0
            pv.append(res.wald_test(R, scalar=True).pvalue)

        out.append({
            "model": f"{family}-{size}B",
            "delta Apple": bl["delta"]["brand_Apple"],
            "B[screen=14-inch]": bl["B[screen=14-inch]"]["brand_Apple"],
            "B[ram=8GB]": bl["B[ram=8GB]"]["brand_Apple"],
            "net under ram contract": bl["delta"]["brand_Apple"] + bl["B[ram=8GB]"]["brand_Apple"],
            "p(screen leak=0)": pv[0], "p(ram leak=0)": pv[1],
        })
    return pd.DataFrame(out).set_index("model")


for spec_tied in (False, True):
    t = leakage_table(spec_tied)
    print(("SPEC-TIED pairs (brand is the only difference)" if spec_tied
           else "ALL pairs"))
    fmt = t.copy()
    for c in fmt.columns:
        fmt[c] = fmt[c].map("{:.2e}".format if c.startswith("p(") else "{:.2f}".format)
    print(fmt.to_string(), "\n")

**The leakage is negative in 8/8 models for both contracts.** And on the spec-tied stratum the
`delta` column reproduces the "same ram + same screen" numbers of the 2026-08-10 entry exactly
(qwen-32B 7.31, gemma-27B 10.79) — same measurement, new parameterization.

Look at `net under ram contract` = `delta + B[ram=8GB]`. It sits near **zero** for every capable
model. The contract does not shrink Apple's premium; it **cancels** it.

On the p-values: with ~990 clusters even a 0.04 coefficient comes out at `<1e-16` (see
qwen-0.5B, whose brand spread is 0.13 — pure noise). Significance is not the interesting
quantity here; the effect size relative to `delta` is. Which is the next cell.

## 4. One number per contract: how much of the preference does it cancel?

Regress the leakage row on `delta` over the whole 5-brand block. Both are sum-to-zero, so the
slope is gauge-safe and scale-free.

    slope = 0    contract leaves the brand preference alone
    slope = -1   contract exactly cancels it

In [ ]:
rows = []
for family, size in MODELS:
    runs = load_runs(family, size)
    if len(runs) < 4:
        continue
    res, conds, zn = fit_grum(runs, features=["brand"], spec_tied=True)
    bl = blocks(res, conds, zn, ["brand"])
    d = np.array([bl["delta"][f"brand_{b}"] for b in BRAND])
    r = {"model": f"{family}-{size}B", "brand spread": d.max() - d.min()}
    for (f, lv) in X_COLS:
        v = np.array([bl[f"B[{f}={lv}]"][f"brand_{b}"] for b in BRAND])
        r[f"contract on {f}"] = float(v @ d / (d @ d))
    rows.append(r)

canc = pd.DataFrame(rows).set_index("model")
print(canc.round(2).to_string())
print("\nmean over the 7 models with real signal (qwen-0.5B excluded):")
print(canc.drop(index="qwen-0.5B")[["contract on screen", "contract on ram"]].mean().round(2).to_string())

**A ram contract cancels ~94% of the brand preference. A screen contract cancels ~54%.**

That is C6 plus the C5 ram/screen asymmetry as two numbers, replacing a table of correlations.
It also fits the lexicographic reading: naming ram promotes an **absolute veto** above brand,
naming screen promotes a merely **partial** tendency.

Prediction for the runs proposed below: `ram=4GB` and `ram=16GB` should also land near -0.94,
and `screen=13-inch` / `screen=16-inch` near -0.54. If instead the slope tracks *how much the
contract asks the model to give up*, `ram=16GB` (where contract and preference agree) will come
out much closer to 0.

## 5. Does it GENERALIZE? — predicting a contract we never ran

Everything above is in-sample. A saturated `x` reproduces every condition perfectly and predicts
**nothing** — which makes it a description, not a model.

The current data supports exactly one held-out test: fit on `{none, screen=14, ram=8}`, predict
`screen=14 + ram=8`. Additivity in `x` forces the prediction, with no free parameters:

    S      = w(14+8) - w(none)                     the true shift
    S_hat  = (w(14) - w(none)) + (w(8) - w(none))  predicted from the singles alone

    gen R2 = 1 - ||S - S_hat||^2 / ||S||^2

In [ ]:
DOUBLE = "ram=8GB+screen=14-inch"

def per_condition_w(runs, features, spec_tied):
    """One BT fit per condition, returned as a dict of level -> weight."""
    out = {}
    for cid, (_, df) in runs.items():
        if spec_tied:
            df = df[(df.a_screen == df.b_screen) & (df.a_ram == df.b_ram)]
        dz = np.hstack([effects_code(df[f"a_{f}"], LEVELS[f])
                        - effects_code(df[f"b_{f}"], LEVELS[f]) for f in features])
        r = sm.OLS((df.score_a - df.score_b).values, sm.add_constant(dz)).fit()
        out[cid] = restore_dropped(r.params[1:], features)
    return out


def shifts(spec_tied):
    """(true shift, additive prediction) per model, as aligned vectors."""
    features = ["brand"] if spec_tied else ["brand", "screen", "ram"]
    order = [f"{f}_{lv}" for f in features for lv in LEVELS[f]]
    out = {}
    for family, size in MODELS:
        runs = load_runs(family, size)
        if len(runs) < 4:
            continue
        w = per_condition_w(runs, features, spec_tied)
        v = lambda c: np.array([w[c][k] for k in order])
        none = v("none")
        S     = v(DOUBLE) - none
        S_hat = (v("screen=14-inch") - none) + (v("ram=8GB") - none)
        out[f"{family}-{size}B"] = (S, S_hat)
    return out


for spec_tied in (False, True):
    D = shifts(spec_tied)
    rows = []
    for m, (S, H) in D.items():
        rows.append({"model": m,
                     "||S|| true": np.linalg.norm(S),
                     "||S_hat|| pred": np.linalg.norm(H),
                     "gen R2": 1 - ((S - H) ** 2).sum() / (S ** 2).sum(),
                     "slope": float(H @ S / (H @ H))})
    t = pd.DataFrame(rows).set_index("model")
    print("brand block, spec-tied pairs" if spec_tied else "all features, all pairs")
    print(t.round(3).to_string())
    print("   mean over the 7 real models:",
          t.drop(index="qwen-0.5B")[["gen R2", "slope"]].mean().round(3).to_dict(), "\n")

Two results:

- **The direction generalizes.** `gen R2 ~ 0.75` for a contract with *zero* parameters fitted on
  it. Three conditions predict a fourth.
- **The magnitude over-shoots.** `slope < 1` in 7 of 8 models and `||S_hat|| > ||S||` almost
  everywhere: **two contracts together do less than the sum of what each does alone.**

Sub-additivity, not the "conjunction bonus" an earlier draft assumed.

### 5.1 One shared scalar fixes the magnitude

Rather than a free conjunction vector per double (312 parameters, transfers to nothing), shrink
the shift by a single global factor:

    w_i = a + lambda^(q_i - 1) * B' x_i        q_i = how many features the contract names

`lambda` is fitted once, shared across **all** models. The "ceiling" column is what a per-model
rescaling could achieve — the best any pure-magnitude correction can do.

In [ ]:
for spec_tied in (False, True):
    D = shifts(spec_tied)
    D = {m: v for m, v in D.items() if m != "qwen-0.5B"}     # no signal to rescale
    S_all = np.concatenate([s for s, _ in D.values()])
    H_all = np.concatenate([h for _, h in D.values()])
    lam = float(H_all @ S_all / (H_all @ H_all))

    rows = []
    for m, (S, H) in D.items():
        rows.append({
            "model": m,
            "additive": 1 - ((S - H) ** 2).sum() / (S ** 2).sum(),
            f"x lambda={lam:.2f}": 1 - ((S - lam * H) ** 2).sum() / (S ** 2).sum(),
            "ceiling": float(np.corrcoef(S, H)[0, 1] ** 2),
        })
    t = pd.DataFrame(rows).set_index("model")
    print("brand block, spec-tied pairs" if spec_tied else "all features, all pairs")
    print(f"   one global saturation factor lambda = {lam:.3f}")
    print(t.round(3).to_string())
    print("   mean:", t.mean().round(3).to_dict(), "\n")

**One parameter takes brand-block generalization from 0.77 to 0.97** — essentially the ceiling
(0.98). That is the whole difference between describing and predicting: a free conjunction vector
per double costs 312 parameters and transfers to nothing; `lambda` costs one and transfers to all
39 doubles.

**Caveat.** We only observe `q` in {1, 2}, so the *functional form* is not identified.
`lambda^(q-1)` and `q^(-alpha)` fit identically here and diverge at `q = 3` (0.50 vs 0.58).
**One triple contract settles it.**

### 5.2 Why it generalizes: the leakage has no shape of its own

A free `B` needs one row per (feature, level), so a contract level never run has no row at all.
But the fitted rows turn out to be nearly **parallel** — and parallel to `-delta`.

In [ ]:
cos = lambda u, v: float(u @ v / np.linalg.norm(u) / np.linalg.norm(v))
rows = []
for family, size in REAL:
    runs = load_runs(family, size)
    if len(runs) < 4:
        continue
    res, conds, zn = fit_grum(runs, features=["brand"], spec_tied=True)
    bl = blocks(res, conds, zn, ["brand"])
    vec = lambda k: np.array([bl[k][f"brand_{b}"] for b in BRAND])
    d, bs, br = vec("delta"), vec("B[screen=14-inch]"), vec("B[ram=8GB]")
    rows.append({"model": f"{family}-{size}B",
                 "cos(B_screen, B_ram)": cos(bs, br),
                 "cos(B_screen, -delta)": cos(bs, -d),
                 "cos(B_ram, -delta)": cos(br, -d)})
t = pd.DataFrame(rows).set_index("model")
print(t.round(3).to_string())
print("\nmean:", t.mean().round(3).to_dict())

Six of seven models are at **0.97+**. Leakage does not have its own shape: **every contract
shrinks the brand preference along `-delta`, its own direction**, and contracts differ only in
*how much*. (gemma-1B at 0.54 is the exception — also the weakest signal.)

That licenses the structured, generalizing form — for a feature `g` the contract does **not**
name:

    w_g(x) = (1 - kappa_g(x)) * a_g,    kappa_g(x) = lambda^(q-1) * sum over named f of kappa_{f->g}

with `kappa_{ram->brand} = 0.94`, `kappa_{screen->brand} = 0.54`. Check:
`0.71 * (0.94 + 0.54) = 1.05` — the double contract cancels the brand preference and very
slightly reverses it, which is what the `net under ram contract` column showed.

**The load-bearing assumption is that `kappa_{f->g}` depends on the feature `f`, not on the
level.** If true, `ram=4GB` tells you what `ram=16GB` does and 11 rows collapse to 6 scalars.
**It is completely untested** — every constrained feature has been run at exactly one level.
Testing it is what Phase A below is for.

## 6. Sampling: contract space is exponential, parameters need not be

In [ ]:
from itertools import product
from math import prod
from collections import Counter

NLEV = {f: len(LEVELS[f]) for f in LEVELS}
space = prod(v + 1 for v in NLEV.values())
print(f"contract space = {' x '.join(str(v + 1) for v in NLEV.values())} = {space}")
cnt = Counter()
for combo in product(*[[None] + list(range(NLEV[f])) for f in NLEV]):
    cnt[sum(c is not None for c in combo)] += 1
for q in sorted(cnt):
    print(f"   naming {q} feature(s): {cnt[q]:>3}")
print(f"   collected so far: 4  ({4 / space:.0%})\n")

nz, sumL, F = 8, sum(NLEV.values()), len(NLEV)
print("parameters to cover all 96 contracts:")
print(f"   saturated, one w per contract      : {space * sumL:>5}   hopeless")
print(f"   GRUM, free B main effects          : {sumL * nz:>5}")
print(f"      + free conjunction per double   : {39 * nz:>5}   does NOT generalize")
print(f"   structured B + saturation          : {nz + F + F * (F - 1) + 1:>5}"
      f"   = a({nz}) + rho({F}) + kappa({F * (F - 1)}) + lambda(1)")
print("\n   space grows as PROD(L_f + 1); structured params grow as F^2 + sum(L_f).")

L2 = dict(NLEV, price=4); s2 = prod(v + 1 for v in L2.values()); F2 = len(L2)
nz2 = sum(L2.values()) - F2
print(f"\n   add a 4th feature (4 price levels): space {space} -> {s2}, "
      f"structured params {nz + F + F * (F - 1) + 1} -> {nz2 + F2 + F2 * (F2 - 1) + 1}")

So the sampling rule follows from the counting:

> **Run the singles exhaustively** — there are only `sum(L_f) = 11` of them and they carry all of
> `B`. **Sample the combinations** — there are 84 of them, and predicting those is the model's job.

Priority order, and what each phase actually buys:

| phase | contracts | conditions | runs (x8) | buys |
|---|---|---|---|---|
| **A** | `screen=13`, `screen=16`, `ram=4`, `ram=16` | 4 | 32 | **tests whether `kappa` is level-independent** — the assumption §5.2 rests on |
| **B** | `brand=Apple` + 2-3 others | 3-4 | 24-32 | the symmetric question: does a *brand* contract leak into specs? |
| **C** | 4-6 doubles across all 3 feature-pairs, + 2 triples | 6-8 | 48-64 | fits `lambda` and **identifies its form** at `q=3` |
| **D** | ~8 contracts sampled at random, never fitted | 8 | 64 | honest held-out generalization number |

**Phase A first.** Cheapest, and if `kappa` turns out level-dependent the 18-parameter model
collapses and everything downstream changes.